# 機械学習演習 第6回 問題1 解答

問題1「単回帰（正規方程式による解）」の解答である。

このNotebookでは、乱数シードを固定して人工データを生成し、`LinearRegression`による最小二乗解、可視化、`MinMaxScaler`によるスケーリングの影響、ブートストラップ法による係数$a$の95%信頼区間を確認する。

## 事前準備

必要なライブラリを読み込む。乱数シードを固定することで、実行するたびに同じデータと結果が得られるようにする。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler

# グラフ表示の設定
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

# 再現性のための乱数シード
SEED = 42
rng = np.random.default_rng(SEED)

## 1. データセット $(X, y)$ の作成

特徴量$x_i$を区間$[0.0, 1.0 \times 10^8)$の一様乱数で100個生成する。

目的変数は

$$y_i = ax_i + b + c_i$$

で作る。ここで、$a = 0.002$、$b = 5.0 \times 10^7$、$c_i$は平均0、標準偏差$1.0 \times 10^4$の正規乱数である。

In [ ]:
# データ数
N = 100

# 真のパラメータ
a_true = 0.002
b_true = 5.0e7
sigma = 1.0e4

# 特徴量 X: sklearnに渡しやすいように shape=(N, 1) にする
X = rng.uniform(0.0, 1.0e8, size=(N, 1))

# ノイズ c_i
c = rng.normal(loc=0.0, scale=sigma, size=N)

# 目的変数 y
y = a_true * X.ravel() + b_true + c

print("X.shape =", X.shape)
print("y.shape =", y.shape)
print("Xの先頭5件:", X[:5].ravel())
print("yの先頭5件:", y[:5])

**解説**

`LinearRegression`などのscikit-learnのモデルでは、特徴量は通常2次元配列として渡す。そのため、単回帰で説明変数が1つだけの場合でも、`X`は`(100, 1)`の形にしている。

`y`は1次元配列で問題ない。真の関係は傾き0.002、切片$5.0 \times 10^7$だが、正規乱数のノイズを足しているため、推定される係数は真の値と完全には一致しない。

## 2. `LinearRegression`による単回帰式と決定係数$R^2$

`LinearRegression`を使って最小二乗解を求める。問題文では「正規方程式で得られるものと同じ最小二乗解」と説明されている。

In [ ]:
model = LinearRegression()
model.fit(X, y)

a_hat = model.coef_[0]
b_hat = model.intercept_
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)

print(f"推定された単回帰式: y = {a_hat:.8f} x + {b_hat:.3f}")
print(f"真の単回帰式      : y = {a_true:.8f} x + {b_true:.3f}")
print(f"決定係数 R^2      : {r2:.6f}")

**解説**

`model.coef_[0]`が傾き$a$の推定値、`model.intercept_`が切片$b$の推定値である。

このNotebookの固定シードでは、おおよそ

$$\hat{y} = 0.00200766x + 49999486.997$$

となり、決定係数は$R^2 \simeq 0.968902$である。

今回のデータは真の直線に標準偏差$1.0 \times 10^4$のノイズを加えて作られているため、推定値は真の$a=0.002$、$b=5.0 \times 10^7$の近くになる。

決定係数$R^2$は、モデルが目的変数のばらつきをどれだけ説明できているかを表す値である。1に近いほどよく当てはまっている。

## 3. データと単回帰式の可視化

散布図でデータを表示し、その上に求めた回帰直線を重ねる。

In [ ]:
x_grid = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
y_grid = model.predict(x_grid)

plt.scatter(X.ravel(), y, alpha=0.75, label="data")
plt.plot(x_grid.ravel(), y_grid, color="red", linewidth=2, label="LinearRegression")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Simple Linear Regression")
plt.legend()
plt.show()

**解説**

散布図の点はノイズを含む観測データである。赤い直線は最小二乗法で求めた回帰直線である。

ノイズがあるためすべての点が直線上に乗るわけではないが、全体として右上がりの直線関係が確認できる。

## 4. `MinMaxScaler`で特徴量をスケーリングしてから単回帰

特徴量$x$を`MinMaxScaler`で$[0, 1]$の範囲に変換してから、もう一度`LinearRegression`を適用する。

正規方程式に対応する最小二乗解では、スケーリングしても元の$x$に対する予測値は同じになるため、決定係数$R^2$も変わらないことを確認する。

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

model_scaled = LinearRegression()
model_scaled.fit(X_scaled, y)

y_pred_scaled = model_scaled.predict(X_scaled)
r2_scaled = r2_score(y, y_pred_scaled)

# スケーリング後の変数 z = (x - x_min) / (x_max - x_min) に対する係数
a_scaled = model_scaled.coef_[0]
b_scaled = model_scaled.intercept_

# 元の x に対する係数へ変換
x_min = scaler.data_min_[0]
x_range = scaler.data_range_[0]
a_scaled_back = a_scaled / x_range
b_scaled_back = b_scaled - a_scaled_back * x_min

print("スケーリング前")
print(f"  y = {a_hat:.8f} x + {b_hat:.3f}")
print(f"  R^2 = {r2:.12f}")

print("\nスケーリング後の変数 z に対する式")
print(f"  y = {a_scaled:.3f} z + {b_scaled:.3f}")
print(f"  R^2 = {r2_scaled:.12f}")

print("\nスケーリング後の式を元の x スケールへ戻した式")
print(f"  y = {a_scaled_back:.8f} x + {b_scaled_back:.3f}")

print("\nR^2は一致しているか:", np.isclose(r2, r2_scaled))

**解説**

`MinMaxScaler`は$x$を

$$z = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

に変換する。スケーリング後のモデルは$z$に対して

$$y = A z + B$$

という形になる。これを元の$x$に戻すと

$$y = \frac{A}{x_{\max}-x_{\min}}x + \left(B - \frac{A}{x_{\max}-x_{\min}}x_{\min}\right)$$

である。

このNotebookの固定シードでは、スケーリング前後の決定係数はどちらも$R^2 \simeq 0.968902481095$となり、一致する。

つまり、説明変数のスケールを変えても、表現している直線は同じである。そのため、各データ点に対する予測値は同じになり、決定係数$R^2$も変わらない。

この点が、後の問題で扱う勾配法と異なる。正規方程式に対応する最小二乗解では直接解が求まるため、特徴量のスケールによる最適化の進みやすさの影響を受けない。

スケーリング後のモデルで得られる回帰直線も、元の$x$軸上に描くとスケーリング前と同じ直線になる。

In [ ]:
x_grid_scaled = scaler.transform(x_grid)
y_grid_scaled = model_scaled.predict(x_grid_scaled)

plt.scatter(X.ravel(), y, alpha=0.75, label="data")
plt.plot(x_grid.ravel(), y_grid, color="red", linewidth=2, label="before scaling")
plt.plot(x_grid.ravel(), y_grid_scaled, color="blue", linestyle="--", linewidth=2, label="after scaling")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Regression Lines Before and After MinMax Scaling")
plt.legend()
plt.show()

## 5. ブートストラップ法による係数$a$の95%信頼区間

ブートストラップ法では、元の100個のデータから重複を許して100個のデータを取り出し、そのリサンプルデータに対して回帰係数$a$を推定する。

これを2000回繰り返し、得られた$a$の分布の2.5パーセンタイルと97.5パーセンタイルを95%信頼区間とする。

In [ ]:
n_bootstrap = 2000
bootstrap_rng = np.random.default_rng(SEED + 1)

a_bootstrap = np.empty(n_bootstrap)

for i in range(n_bootstrap):
    sample_idx = bootstrap_rng.integers(0, N, size=N)
    X_sample = X[sample_idx]
    y_sample = y[sample_idx]

    boot_model = LinearRegression()
    boot_model.fit(X_sample, y_sample)
    a_bootstrap[i] = boot_model.coef_[0]

ci_lower, ci_upper = np.percentile(a_bootstrap, [2.5, 97.5])
contains_true = ci_lower <= a_true <= ci_upper

print(f"元データから推定した a: {a_hat:.8f}")
print(f"ブートストラップ平均: {a_bootstrap.mean():.8f}")
print(f"95%信頼区間: [{ci_lower:.8f}, {ci_upper:.8f}]")
print(f"真の値 a = {a_true:.8f} は95%信頼区間に含まれるか: {contains_true}")

In [ ]:
plt.hist(a_bootstrap, bins=40, alpha=0.75, edgecolor="black")
plt.axvline(a_true, color="red", linestyle="-", linewidth=2, label="true a")
plt.axvline(ci_lower, color="black", linestyle="--", linewidth=2, label="95% CI")
plt.axvline(ci_upper, color="black", linestyle="--", linewidth=2)
plt.xlabel("estimated coefficient a")
plt.ylabel("frequency")
plt.title("Bootstrap Distribution of Coefficient a")
plt.legend()
plt.show()

**解説**

ブートストラップ法では、標本を何度も取り直したと仮定して推定量のばらつきを調べる。ここでは各リサンプルで傾き$a$を推定し、その分布から95%信頼区間を求めた。

このNotebookの固定シードでは、係数$a$の95%信頼区間はおおよそ

$$[0.00194443, 0.00207956]$$

となる。

出力された`contains_true`が`True`であれば、真の値$a=0.002$は95%信頼区間に含まれている。

ただし、信頼区間は乱数で生成したデータに依存する。別の乱数シードでデータを作ると、推定値や信頼区間は少し変わる。

## まとめ

- 人工データは、$y_i = 0.002x_i + 5.0 \times 10^7 + c_i$に従って生成した。
- `LinearRegression`により、真の係数に近い単回帰式が得られた。
- 散布図と回帰直線から、データ全体に直線関係があることを確認した。
- `MinMaxScaler`で特徴量をスケーリングしても、正規方程式に対応する最小二乗解では予測値と$R^2$は変わらなかった。
- ブートストラップ法を2000回行い、係数$a$の95%信頼区間を求め、真の値$a=0.002$が含まれるか確認した。